# Stickman Thoughts — 30-scene Space Adventure

Generates 30 separate 9:16 story frames with the pinned FLUX.2 Klein 4B model on Kaggle. Every frame is reference-conditioned using the approved astronaut costume reference from the owner's `yt-auto` repository. This is a visual consistency experiment; inspect every image before use.

The notebook fails closed if the installed pipeline does not explicitly support image conditioning. It never silently falls back to text-to-image.


In [ ]:
import os, sys, json, time, subprocess, pathlib, hashlib, traceback, uuid, gc, inspect, urllib.request
from PIL import Image, ImageDraw
import numpy as np
RUN_ID=globals().get('RUN_ID','manual-'+uuid.uuid4().hex); RUN_TOKEN=globals().get('RUN_TOKEN','manual'); COMMIT_SHA=globals().get('COMMIT_SHA','unknown'); WORKFLOW_RUN_ID=globals().get('WORKFLOW_RUN_ID','local')
OUT=pathlib.Path('/kaggle/working/outputs'); OUT.mkdir(parents=True,exist_ok=True)
os.environ.setdefault('HF_HOME','/tmp/hf_home'); os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
MODEL_ID='black-forest-labs/FLUX.2-klein-4B'; MODEL_REVISION='e7b7dc27f91deacad38e78976d1f2b499d76a294'
WIDTH,HEIGHT,STEPS,GUIDANCE=576,1024,4,1.0
PIP_PACKAGES=['diffusers==0.42.1','transformers==5.22.1','huggingface-hub==1.33.0','accelerate','psutil','pillow']
BASE='Keep the exact approved Stickman Thoughts mascot identity from the reference: large circular white head, thick dark outline, same small oval eyes, eyebrows and mouth, thin simple stick limbs and the same simple body proportions. The character wears the same consistent white astronaut spacesuit with restrained blue and orange details and a clear astronaut helmet. Flat 2D vector cartoon, clean thick outlines, soft muted colours, simple shading, readable expressive pose. Do not redesign the mascot, change its face, add extra fingers, or change the spacesuit between scenes. Compose as one complete vertical 9:16 illustration with a detailed coherent environment, cinematic depth and clear foreground, middle ground and background. No text, labels, panel borders, collage, watermark or captions.'
SCENES=[["01_earth_plan","On Earth at a small spaceport viewing area, the Stickman astronaut stands beside a window looking toward a tall rocket on its launch tower. Blue sky, distant launch facility, green grass, hopeful anticipation."],["02_suit_preparation","Inside a clean astronaut preparation room, the Stickman puts on and checks the same white astronaut spacesuit with simple blue and orange details, helmet nearby, equipment lockers and mission checklist in the background."],["03_walk_to_launchpad","The Stickman astronaut walks along a marked path toward a large white rocket on a launchpad. Wide launch facility, service towers, railings, bright morning sky."],["04_enter_rocket","The Stickman astronaut climbs the access gantry and enters the open hatch of the rocket. Show the rocket body towering beside the gantry, Earth launch facility in the background."],["05_inside_rocket","Inside the compact rocket cockpit, the Stickman astronaut is strapped safely into the pilot seat, hands near control panels, switches and small screens around the cabin."],["06_countdown","Close interior view of the Stickman astronaut in the cockpit during final countdown, focused expression, illuminated control panels and a large countdown display, rocket still on launchpad."],["07_liftoff","Exterior wide shot of the rocket lifting off from Earth, powerful bright engines, billowing smoke and launch tower below, blue sky, dramatic but family-friendly."],["08_through_clouds","The rocket climbs through thick white clouds, sunlight breaking through, Earth atmosphere visible around it, rocket angled upward with a bright engine trail."],["09_leave_atmosphere","The rocket crosses the edge of Earth's atmosphere into the dark of space, curved blue atmospheric horizon below, first stars appearing."],["10_earth_behind","Wide cinematic space view of the rocket travelling away from Earth, beautiful blue planet and thin glowing atmosphere behind it, deep black space."],["11_sun_and_stars","The rocket cruises through space with the bright Sun shining off to one side, many distant stars and a few planets visible, Earth now smaller in the distance."],["12_weightlessness","Inside the rocket cabin, the Stickman astronaut floats gently in zero gravity, a small tool and checklist floating nearby, cockpit windows show stars."],["13_turbulence","Inside the cockpit, the rocket shakes from unexpected turbulence, warning lights blink, the Stickman astronaut grips the controls with a worried expression."],["14_stabilize_rocket","The Stickman astronaut struggles to steer the shaking rocket, leaning forward and carefully operating the controls, warning indicators and starfield visible through the window."],["15_system_stable","The rocket becomes steady again; the Stickman astronaut relaxes and gives a relieved thumbs-up, cockpit warning lights turn calm, stars visible outside."],["16_radar_signal","The Stickman astronaut notices an unusual signal on the cockpit radar screen, a mysterious small planet icon glowing on the display, surprised expression."],["17_alien_planet_first_sight","Through the rocket window, the Stickman astronaut sees a beautiful unfamiliar alien planet with purple-pink skies, rings, distant moons, mountains and glowing blue terrain."],["18_approach_planet","Exterior view of the rocket descending toward the alien planet, curved horizon, layered purple clouds, strange mountains and glowing rivers below."],["19_land_on_planet","The rocket makes a careful landing on a flat rocky alien plain, small dust clouds around the landing legs, dramatic alien mountains and a ringed moon in the sky."],["20_step_out","The Stickman astronaut steps down from the landed rocket onto the alien planet, wearing the same spacesuit and helmet, looking around cautiously at the strange landscape."],["21_explore_alien_world","The Stickman astronaut walks across the alien planet exploring, unusual rock formations, glowing plants, blue crystals and distant purple mountains."],["22_crystal_discovery","The Stickman astronaut discovers a cluster of tall glowing blue crystals on the alien surface, kneeling to examine them, colorful alien plants and ringed sky behind."],["23_aliens_in_distance","The Stickman astronaut notices several small green alien creatures watching from behind rocks in the distance, mysterious but not yet threatening, wide alien landscape."],["24_first_contact","The Stickman astronaut cautiously raises one hand in greeting toward the green aliens, who stand together nearby, strange plants and the landed rocket visible in the distance."],["25_aliens_turn_hostile","The green aliens suddenly look angry and move toward the Stickman astronaut, who steps backward in alarm, tense action composition, no weapons or gore."],["26_aliens_attack","The aliens rush toward the Stickman astronaut across the rocky ground, the astronaut turns to flee, dynamic action, dust and glowing plants, family-friendly peril without violence or injury."],["27_run_for_rocket","The Stickman astronaut runs urgently toward the distant rocket, looking back fearfully as the green aliens chase behind, wide view of alien plain and mountains."],["28_reach_rocket","The Stickman astronaut reaches the rocket access ladder and climbs quickly toward the hatch, aliens approaching in the background, dramatic low-angle view."],["29_escape_takeoff","The rocket blasts off from the alien planet as the Stickman astronaut escapes, bright engine flames, dust cloud, green aliens left safely on the ground, ringed planet sky."],["30_leave_planet","Wide final space shot of the rocket speeding away from the alien planet toward the stars, alien world shrinking below, Sun glinting in the distance, triumphant escape mood."]]
CASES=[{'name':name,'prompt':BASE+' Scene description: '+desc,'seed':52000+i} for i,(name,desc) in enumerate(SCENES)]
REPORT={'run_token':RUN_TOKEN,'run_id':RUN_ID,'commit_sha':COMMIT_SHA,'workflow_run_id':WORKFLOW_RUN_ID,'status':'running','error_message':None,'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'install_ok':False,'model_downloaded':False,'pipeline_loaded':False,'cuda_available':False,'reference_conditioning_supported':False,'expected_total_images':30,'results':[]}
STARTED=time.monotonic()


In [ ]:
REF_URL='https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_astronaut.png'
REF_PATH=pathlib.Path('/kaggle/working/approved_astronaut_reference.png')
urllib.request.urlretrieve(REF_URL,REF_PATH)
with Image.open(REF_PATH) as im: im.verify()
def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
 return h.hexdigest()
REPORT['reference_image_sha256']=sha256(REF_PATH)
print('Approved astronaut reference downloaded:',REF_PATH,'SHA256:',REPORT['reference_image_sha256'])


In [ ]:
def sh(cmd): return subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
def install():
 p=sh([sys.executable,'-m','pip','install','--no-cache-dir']+PIP_PACKAGES); (OUT/'install.log').write_text(p.stdout,encoding='utf-8')
 if p.returncode: raise RuntimeError('pip install failed: '+p.stdout[-2500:])
 import torch, diffusers, transformers
 REPORT['torch_version']=torch.__version__; REPORT['diffusers_version']=diffusers.__version__; REPORT['transformers_version']=transformers.__version__
 REPORT['cuda_available']=bool(torch.cuda.is_available()); REPORT['gpu_names']=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
 REPORT['gpu_total_vram_bytes']=[int(torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
 if not REPORT['cuda_available']: raise RuntimeError('CUDA unavailable; enable Kaggle GPU accelerator')
 REPORT['install_ok']=True
def download():
 from huggingface_hub import snapshot_download
 t=time.monotonic(); p=snapshot_download(MODEL_ID,revision=MODEL_REVISION); REPORT['download_seconds']=round(time.monotonic()-t,1); REPORT['model_downloaded']=True; return p
def load(path):
 import torch
 from diffusers import Flux2KleinPipeline
 sig=inspect.signature(Flux2KleinPipeline.__call__); REPORT['pipeline_call_parameters']=list(sig.parameters.keys())
 if 'image' not in sig.parameters: raise RuntimeError('Installed Flux2KleinPipeline.__call__ has no image parameter; refusing to fake reference conditioning')
 REPORT['reference_conditioning_supported']=True
 t=time.monotonic(); pipe=Flux2KleinPipeline.from_pretrained(path,torch_dtype=torch.float16); pipe.enable_model_cpu_offload(gpu_id=0)
 REPORT['pipeline_loaded']=True; REPORT['pipeline_load_seconds']=round(time.monotonic()-t,1); return pipe
def save_result(img,case,elapsed):
 path=OUT/('klein4b_'+case['name']+'.png'); img.save(path)
 std=float(np.asarray(img.convert('L'),dtype='float32').std())
 item={'case':case['name'],'scene_description':SCENES[len(REPORT['results'])][1],'seed':case['seed'],'file':path.name,'seconds':elapsed,'image_width':img.width,'image_height':img.height,'image_sha256':sha256(path),'source_reference_sha256':REPORT['reference_image_sha256'],'looks_broken':std<8}
 REPORT['results'].append(item)
def make_contact_sheet():
 files=sorted(OUT.glob('klein4b_*.png')); cols=5; tile_w,tile_h=160,306; rows=(len(files)+cols-1)//cols
 sheet=Image.new('RGB',(cols*tile_w,rows*tile_h),(235,232,224))
 for i,p in enumerate(files):
  im=Image.open(p).convert('RGB'); im.thumbnail((150,266)); tile=Image.new('RGB',(tile_w,tile_h),'white'); tile.paste(im,((tile_w-im.width)//2,4)); ImageDraw.Draw(tile).text((5,276),p.stem.replace('klein4b_','')[:24],fill='black'); sheet.paste(tile,((i%cols)*tile_w,(i//cols)*tile_h))
 sheet.save(OUT/'klein4b_space_adventure_contact_sheet.png')


In [ ]:
try:
 install(); model_path=download(); pipe=load(model_path)
 import torch
 reference=Image.open(REF_PATH).convert('RGB')
 for case in CASES:
  torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t=time.monotonic()
  gen=torch.Generator(device='cpu').manual_seed(case['seed'])
  img=pipe(prompt=case['prompt'],image=reference,width=WIDTH,height=HEIGHT,num_inference_steps=STEPS,guidance_scale=GUIDANCE,generator=gen).images[0]
  torch.cuda.synchronize(); elapsed=round(time.monotonic()-t,2); save_result(img,case,elapsed)
  print('Generated',case['name'],'in',elapsed,'seconds')
  del img; gc.collect(); torch.cuda.empty_cache()
 REPORT['all_pictures_ok']=len(REPORT['results'])==30 and all(not x['looks_broken'] for x in REPORT['results'])
 REPORT['status']='success' if REPORT['all_pictures_ok'] else 'failed'
 if REPORT['status']!='success': REPORT['error_message']='Expected 30 valid scene images; actual count or image quality check failed.'
except Exception as exc:
 REPORT['status']='failed'; REPORT['error_message']=f'{type(exc).__name__}: {exc}'; traceback.print_exc()
finally:
 REPORT['notebook_duration_seconds']=round(time.monotonic()-STARTED,2)
 try: make_contact_sheet()
 except Exception: traceback.print_exc()
 (OUT/'benchmark_report.json').write_text(json.dumps(REPORT,indent=2),encoding='utf-8')
 print(json.dumps(REPORT,indent=2))


In [ ]:
from IPython.display import display, Image as IPyImage
contact=OUT/'klein4b_space_adventure_contact_sheet.png'
if contact.exists(): display(IPyImage(filename=str(contact)))
print('Space adventure status:',REPORT.get('status'),'images:',len(REPORT.get('results',[])))
print('All 30 scenes require human visual review before publication.')
